# 策略概述

**SSD-DTW-PCA** 同時採用兩種走勢相似度——**SSD**（逐日對齊的距離）與 **DTW**（容忍時間錯位的距離）——再用主成分分析把兩者**融合成單一分數**排序，兼取兩種距離的優點。

這是本研究參考的論文方法（許鈞翔 2025）之主要距離排序基準：先在同產業內通過共整合檢定，再以融合分數挑最佳配對。


# 策略架構

四層管線與 DTW 版相同，差別在**排序層把 SSD 與 DTW 兩種距離用主成分融合成一個分數**。

```{mermaid}
flowchart LR
  P["形成期日價格<br/>標準化"] --> G["分組<br/>GICS 產業"]
  G --> F["篩選<br/>共整合 + 半衰期 + Hurst"]
  F --> R["排序<br/>SSD ＋ DTW → PCA 融合分數"]
  R --> T["Top N 配對<br/>→ 交易期"]
```

| 層 | 本策略採用 | 用途 |
| :--- | :--- | :--- |
| 分組 | GICS 產業分類 | 同產業候選 |
| 篩選 | 共整合 + 半衰期 + Hurst | 確認價差均值回歸 |
| 排序 | SSD 與 DTW 經主成分融合（取第一主成分分數） | 綜合兩種距離的相似度 |
| 交易 | Z-Score（標準化空間） | 偏離進場、回歸出場 |


# 參考文獻與引用對應


## 文獻 1：許鈞翔 (2025)

> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。元智大學管理學院財務金融暨會計碩士班碩士論文。

**參考部分**（論文第三章研究方法，實驗組設計）：

- 第一步：Engle-Granger 兩階段檢定（OLS 回歸 → ADF 檢殘差），篩選 $p < 0.01$ 的股票對
- 第二步：對通過共整合的股票對，分別計算 SSD 與 DTW 距離
- **第三步：SSD 與 DTW 標準化後輸入 PCA，依第一主成分（PC1）分數排序選取配對**——本策略的核心排序機制即此實驗組設計

**為何參考**：

- 本策略為此論文**實驗組（SSD + DTW 融合）**的對齊實作：SSD 衡量同步走勢距離、DTW 衡量容許時間錯位的走勢距離，PCA 融合捕捉兩指標的共同變異，避免單一距離度量的偏頗
- 論文以 S&P 500 成分股實證：融合排序在多種交易期與止損條件下顯著優於單獨 SSD，與單獨 DTW 相比亦普遍領先——此為採用融合排序的直接依據



## 文獻 2：Sakoe & Chiba (1978)

> Sakoe, H., & Chiba, S. (1978). Dynamic programming algorithm optimization for spoken word recognition. *IEEE Transactions on Acoustics, Speech, and Signal Processing, 26*(1), 43–49.

**參考部分**：

- DTW 動態規劃遞迴（insertion／deletion／match）與 **Sakoe-Chiba 帶約束**

**為何參考**：

- 融合排序的 DTW 分量以帶約束形式實作（$W = 15$ 天）：限制不合理的長距離時間對齊，並將計算複雜度降至 $O(N \cdot W)$



## 文獻 3：Engle & Granger (1987)

> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica, 55*(2), 251–276.

**參考部分**：

- 兩步驟共整合檢定程序：OLS 估計均衡關係 → 對殘差做 ADF 單根檢定

**為何參考**：

- 階段 3 的「雙向 OLS + ADF」為此程序的實作；對兩個回歸方向各檢定一次、取 p 值較小者，避免因方向選擇錯過共整合配對



## 文獻 4：Krauss, Do & Huck (2016)

> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**參考部分**：

- **OU 半衰期**與 **Hurst 指數**作為均值回歸品質的量化指標

**為何參考**：

- 統計過濾第二道（半衰期 $1$–$42$ 日）與第三道（$H < 0.5$）的門檻設計依據



# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動）內依序執行以下七個階段。
階段 1–5 與 DTW 距離的計算流程共用同一模組（DTW 距離排序流程），差異在階段 6 的排序機制。


## 階段 1：資料範圍界定與產業分組

1. 依 GICS 產業分類將股票分組，配對搜尋只在同產業內進行
2. 產業標記為 `Unknown` 的股票整組跳過
3. 股票數不足 `min_tickers_for_pairing`（= 2）的產業跳過


## 階段 2：對數價格 Z-Score 標準化

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}}{\sigma_{\ln P_i} + \varepsilon}, \qquad \varepsilon = 10^{-12}$$

取對數前以 $\max(P, 10^{-8})$ 下限保護。標準化統計量（`Log_Mean_A/B`、`Log_Std_A/B`）隨配對輸出，供交易期重建同一座標。


## 階段 3：雙向 OLS 回歸與方向決定（依據：Engle & Granger 1987）

對每對股票 $(u, v)$ 兩個方向各做一次 OLS（標準化空間，含截距）：

$$P'_{u,t} = \alpha_1 + \beta_1 P'_{v,t} + \epsilon^{(1)}_t \qquad
P'_{v,t} = \alpha_2 + \beta_2 P'_{u,t} + \epsilon^{(2)}_t$$

分別對殘差做 ADF 檢定（`max_lags=1`），取 p 值較小的方向決定 $(\text{Ticker\_A},\ \text{Ticker\_B})$，
並保留該方向的 $\alpha$（`OLS_Alpha`）、$\beta$（`Hedge_Ratio`）與殘差序列。


## 階段 4：三道統計過濾

| 道次 | 檢定 | 門檻 | 依據 |
| :---: | :--- | :--- | :--- |
| 1 | ADF 共整合 | $p < 0.01$ | Engle & Granger (1987)；許鈞翔 (2025) 採同一顯著水準 |
| 2 | OU 半衰期 | $\lambda < 0$ 且 $1 \le HL \le 42$ 日 | Krauss et al. (2016) |
| 3 | Hurst 指數 | $H < 0.50$（R/S 分析，`already_stationary=True`） | Krauss et al. (2016) |

半衰期：對 $\Delta \epsilon_t = c + \lambda\, \epsilon_{t-1} + u_t$ 最小平方估計，$HL = -\ln 2 / \lambda$。
任一道未通過即淘汰；距離計算只對通過者執行。


## 階段 5：SSD 與 DTW 距離計算

**SSD**（同步逐日比較）：

$$\text{SSD}_{A,B} = \sum_{t=1}^{F} \left(P'_{A,t} - P'_{B,t}\right)^2$$

**Sakoe-Chiba DTW**（容許時間錯位，$W = 15$）：

$$D(i,j) = (P'_{A,i} - P'_{B,j})^2 + \min\big\{ D(i-1,j),\ D(i,j-1),\ D(i-1,j-1) \big\}, \qquad |i - j| \le W$$

兩距離捕捉互補的相似度資訊：SSD 對同步共動敏感，DTW 對存在領先／落後關係的共動敏感。


## 階段 6：PCA 融合排序（依據：許鈞翔 2025 實驗組）

對全部通過檢定的配對：

**步驟 1 — 距離標準化**：

$$Z(\text{SSD}) = \frac{\text{SSD} - \mu_{SSD}}{\sigma_{SSD}}, \qquad Z(\text{DTW}) = \frac{\text{DTW} - \mu_{DTW}}{\sigma_{DTW}}$$

**步驟 2 — PCA 取第一主成分**（`sklearn.decomposition.PCA`，`n_components=1`，`random_state=42`）：

$$\text{PC1}_{A,B} = w_1 \cdot Z(\text{SSD}_{A,B}) + w_2 \cdot Z(\text{DTW}_{A,B})$$

其中 $(w_1, w_2)$ 為第一主成分的 loadings，捕捉兩距離的最大共同變異方向。

**步驟 3 — 方向校正**：若 $w_1 < 0$ 則 PC1 分數取反，確保「距離越小 → 分數越小 → 排名越前」。

**步驟 4 — 選取**：依 PC1 分數**升序**取前 `top_n` 組。

候選對不足 2 組時（PCA 無法擬合），退回依 DTW 距離排序。


## 階段 7：交易期的參數使用方式（座標修正與門檻網格）

**座標修正**：OLS 在標準化空間擬合，config 以 `ignore_ols_alpha=True` 指示交易端忽略 `OLS_Alpha`、直接使用標準化座標重建 spread：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \qquad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio} \cdot P'_{B,t}, \qquad
Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

**門檻網格**（本策略 config 專屬設定）：交易期同時掃描進場門檻與發散停損的組合——

```python
"entry_z_list":        [1.5, 2.0, 2.5],   # 進場 Z 門檻
"dynamic_stop_z_list": [0.0, 3.0, 4.0],   # 發散停損 Z 門檻（0 = 不啟用）
"stop_loss_list":      [0.0],             # 固定不啟用比例停損以控制網格大小
```

形成期統計量整個交易期凍結不變（無前視）。交易決策細節見 `trading/zscore_trading.ipynb`。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗 / 滾動步長 | 252 / 21 交易日 | 全流程輸入 | 約一年 / 一個月 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依融合分數升序取前幾組 |
| 距離衡量 | SSD + DTW（主成分融合） | 排序 | 兩種距離綜合成單一分數 |
| DTW 帶寬 | 15 天 | 排序 | Sakoe-Chiba 限制窗 |
| 共整合顯著水準 | 0.01 | 篩選 | 共整合檢定門檻 |
| 半衰期範圍 | $[1,\ 42]$ 日 | 篩選 | 回歸速度合理區間 |
| Hurst 上限 | 0.50 | 篩選 | 均值回歸判準 |
| 進場門檻網格 | [1.5, 2.0, 2.5] | 交易 | 偏離幾倍標準差進場 |
| 發散停損網格 | [0.0, 3.0, 4.0] | 交易 | 價差續擴時的停損水位 |
